# Notebook 02 — Mixed Effects Model (Approche B) — Couche d'explicabilité

> **Quand utiliser ce notebook ?**  
> Après avoir validé la baseline Transfer Learning (NB 04). Le Mixed Effects n'est pas là pour battre le GBM sur le RMSE — il est là pour **rendre le modèle lisible** en comité de direction.  
>
> *"Le coût de base d'une famille UPS est 1 100€. Cette variante s'en écarte de +80€ pour les condensateurs premium."*  
> → C'est ce que le Mixed Effects permet de dire. Le Transfer Learning seul ne peut pas.
>
> **Prérequis** : avoir exécuté [04_transfer_learning.ipynb](04_transfer_learning.ipynb) et validé que le signal est capturé.

---

## Concepts abordés
1. Effets fixes vs effets aléatoires — intuition
2. Formulation mathématique du modèle
3. Implémentation avec `statsmodels.MixedLM`
4. Lecture des résultats et interprétation métier
5. Comparaison avec un modèle naïf (OLS par variante)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet(Path('data/dataset_industriel.parquet'))
FEATURES = ['composants', 'mo', 'energie', 'volume']
TARGET   = 'cout'
print(f"Dataset : {len(df):,} lignes, {df['variante'].nunique()} variantes")

## 1. Effets fixes vs effets aléatoires : l'intuition

Imagine que tu veux estimer le coût moyen de chaque famille :

**Option A — Effets fixes** : une constante séparée par famille (45 paramètres à estimer)  
→ Précis pour les familles bien représentées, instable pour les familles rares

**Option B — Effets aléatoires** : on suppose que les effets famille suivent une distribution $\mathcal{N}(\mu, \sigma^2)$  
→ Les estimations des familles rares sont **"shrinkées"** vers la moyenne globale $\mu$

Le **modèle à effets mixtes** combine les deux :
- Effets **fixes** pour les features coûts (composants, MO, énergie, volume) — partagés entre tous
- Effets **aléatoires** pour les variantes — chaque variante a une déviation propre, mais contrainte

### Formulation mathématique

$$\text{Coût}_{vt} = \underbrace{\beta_0 + \beta_1 \cdot X_{vt,1} + \ldots + \beta_4 \cdot X_{vt,4}}_{\text{partie fixe}} + \underbrace{u_v}_{\text{effet aléatoire variante}} + \varepsilon_{vt}$$

Avec :
- $u_v \sim \mathcal{N}(0, \sigma_u^2)$ : déviation de la variante $v$ par rapport à la moyenne
- $\varepsilon_{vt} \sim \mathcal{N}(0, \sigma_\varepsilon^2)$ : bruit résiduel

**Le shrinkage émerge naturellement** : pour les variantes avec peu d'observations, $u_v$ sera estimé proche de 0 (pas de déviation significative).

## 2. Implémentation avec statsmodels

In [ ]:
# Ajout d'effets fixes de famille via des dummies
# Le modèle : cout ~ composants + mo + energie + volume + (1|variante)
# avec la famille comme covariable fixe

# Standardisation des features pour faciliter la convergence
df_model = df.copy()
for feat in FEATURES:
    df_model[f'{feat}_std'] = (df[feat] - df[feat].mean()) / df[feat].std()

formula = 'cout ~ composants_std + mo_std + energie_std + volume_std + C(famille)'

model = smf.mixedlm(
    formula=formula,
    data=df_model,
    groups=df_model['variante'],  # effet aléatoire par variante
)

result = model.fit(method='lbfgs', maxiter=200)
print(result.summary())

In [ ]:
# --- Extraction des effets aléatoires (u_v) ---
random_effects = pd.DataFrame({
    'variante'   : list(result.random_effects.keys()),
    'u_v'        : [v['Group'] for v in result.random_effects.values()],
})

# Jointure avec le nombre d'observations
n_obs_df = df.groupby('variante')['cout'].count().reset_index(name='n_obs')
random_effects = random_effects.merge(n_obs_df, on='variante')
random_effects['categorie'] = pd.cut(
    random_effects['n_obs'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print("\nEffets aléatoires par catégorie (u_v) :")
print(random_effects.groupby('categorie')['u_v'].describe().round(1))

In [ ]:
# Visualisation du shrinkage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Effets aléatoires vs n_obs
ax = axes[0]
palette = {'Rare (≤5)': '#e74c3c', 'Interméd. (6-24)': '#f39c12', 'Mature (≥25)': '#27ae60'}
for cat, color in palette.items():
    data = random_effects[random_effects['categorie'] == cat]
    ax.scatter(data['n_obs'], data['u_v'], color=color, alpha=0.7, label=cat, s=60)
ax.axhline(0, color='black', linestyle='--', lw=1.5, label='Shrinkage vers 0')
ax.set_xlabel('Nombre d\'observations')
ax.set_ylabel('Effet aléatoire u_v (€)')
ax.set_title('Shrinkage des effets aléatoires\n(variantes rares → u_v proche de 0)')
ax.legend()

# Distribution des effets aléatoires
ax = axes[1]
for cat, color in palette.items():
    data = random_effects[random_effects['categorie'] == cat]['u_v']
    if len(data) > 0:
        ax.hist(data, bins=20, alpha=0.6, color=color, label=f'{cat} (std={data.std():.1f}€)')
ax.set_xlabel('Effet aléatoire u_v (€)')
ax.set_ylabel('Nombre de variantes')
ax.set_title('Distribution des effets aléatoires\n(plus étroite pour les rares = shrinkage)')
ax.legend()

plt.tight_layout()
plt.savefig('data/fig_mixed_effects_shrinkage.png', dpi=150)
plt.show()

## 3. Interprétation métier

Le résultat clé du modèle mixte est la **séparation des sources de variation** :

- $\sigma_u^2$ : variance entre variantes (hétérogénéité intrinsèque)
- $\sigma_\varepsilon^2$ : variance résiduelle (bruit de mesure, variabilité des commandes)

Le ratio **ICC (Intra-Class Correlation)** mesure quelle part de la variance est due à la variante :

$$\text{ICC} = \frac{\sigma_u^2}{\sigma_u^2 + \sigma_\varepsilon^2}$$

In [ ]:
sigma_u_sq  = result.cov_re.iloc[0, 0]
sigma_eps_sq = result.scale
icc = sigma_u_sq / (sigma_u_sq + sigma_eps_sq)

print(f"Variance inter-variante σ²_u  : {sigma_u_sq:,.0f} €²")
print(f"Variance résiduelle σ²_ε      : {sigma_eps_sq:,.0f} €²")
print(f"ICC (corrélation intra-variante) : {icc:.3f}")
print()
print(f"Interprétation : {icc*100:.1f}% de la variance du coût est due à la variante,")
print(f"et {(1-icc)*100:.1f}% est due au bruit résiduel (fluctuations de commandes).")

## 4. Prédiction et évaluation

In [ ]:
# Prédiction avec les effets aléatoires
df_model['pred_mixedlm'] = result.fittedvalues

# RMSE par catégorie
df_model['categorie'] = pd.cut(
    df_model['n_obs_variante'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

perf = df_model.groupby('categorie').apply(
    lambda g: pd.Series({'rmse': rmse(g['cout'], g['pred_mixedlm']), 'n': len(g)})
).reset_index()

print("Performance Mixed Effects Model par catégorie :")
print(perf.to_string(index=False))

In [ ]:
# Graphique prédit vs réel
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = {'Rare (≤5)': '#e74c3c', 'Interméd. (6-24)': '#f39c12', 'Mature (≥25)': '#27ae60'}

for ax, cat in zip(axes, ['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']):
    data = df_model[df_model['categorie'] == cat]
    ax.scatter(data['cout'], data['pred_mixedlm'],
               alpha=0.4, color=palette[cat], s=30)
    lim = [data[['cout','pred_mixedlm']].min().min(),
           data[['cout','pred_mixedlm']].max().max()]
    ax.plot(lim, lim, 'k--', lw=1.5, label='Prédiction parfaite')
    ax.set_xlabel('Coût réel (€)')
    ax.set_ylabel('Coût prédit (€)')
    ax.set_title(f'{cat}\nRMSE = {rmse(data["cout"], data["pred_mixedlm"]):.1f}€')
    ax.legend(fontsize=8)

plt.suptitle('Mixed Effects Model — Prédit vs Réel', fontsize=13)
plt.tight_layout()
plt.savefig('data/fig_mixedlm_pred_vs_real.png', dpi=150)
plt.show()

## Résumé du Notebook 02

| Aspect | Résultat |
|--------|----------|
| Rôle dans la chaîne | Couche d'explicabilité **après** validation du Transfer Learning |
| Méthode | `statsmodels.MixedLM` — effets fixes familles + effets aléatoires variantes |
| Shrinkage | Les variantes rares ont des u_v proches de 0 → héritent de la moyenne famille |
| ICC | Quantifie la part de variance due à la variante vs le bruit |
| Output clé | "Le coût famille est X€, la variante s'en écarte de u_v€" — lisible en CODIR |
| Limite | Suppose une relation linéaire entre features et coût (moins performant que GBM) |

### Quand passer au Bayésien (NB 03) ?

- Tu as besoin d'**intervalles de confiance formels** sur les prédictions (pas juste un point)
- Certaines familles ont une proportion de variantes rares > 60% et le Mixed Effects converge mal
- Tu veux intégrer de la **connaissance experte** (+15% pour composant premium) de façon formelle

**→ Si la lisibilité suffit, s'arrêter ici.**  
**→ Si IC formels nécessaires : [03_hierarchical_bayesian.ipynb](03_hierarchical_bayesian.ipynb)**